## Step3-Bias-Correct-CMIP

Takes CMIP subglacial discharge and bias-corrects using RACMO subglacial discharge. Plotting code is commented below but could be used to sense-check or debug.

Creator: Donald Slater, donald.slater@ed.ac.uk, 9 July 2025.

Cleaned up by Donald Slater, 9 Feb 2026.

#### Required files/inputs
- RACMO monthly 1958-2024 per-basin subglacial discharge *Q_RACMO_1958_2024.nc* (produced in step 1)
- CMIP monthly per-basin subglacial discharge files, e.g. *Q_CESM2-WACCM_historical_SDBN1_1850_2014.nc*, (produced in step 2)

#### Outputs
- CMIP monthly per-basin bias-corrected subglacial discharge files, e.g. *Q_CESM2-WACCM_historical_SDBN1_1850_2014_biascorrected.nc*
- A record of the bias-correction factors *bias_correction_factors.nc*


### Setup

In [1]:
# RACMO subglacial discharge file from step 1
racmo_runoff_file = '/Users/dslater2/Documents/RACMO/Q_RACMO_1958_2024.nc'

# location of existing CMIP subglacial discharge files from step 2
hist_file = '/Users/dslater2/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1850_2014.nc'
proj_file = '/Users/dslater2/Documents/CESM2-WACCM/ssp585/runoff/projection/Q_CESM2-WACCM_ssp585_SDBN1_2015_2100.nc'
ext_file  = '/Users/dslater2/Documents/CESM2-WACCM/ssp585/runoff/extension/Q_CESM2-WACCM_ssp585_SDBN1_2101_2299.nc'

# location of output bias correction file
bias_file = '/Users/dslater2/Documents/CESM2-WACCM/historical/runoff/bias_correction_factors.nc'

# define bias correction period (inclusive of start and end years) consistent with ocean TF
start_year, end_year = 1985, 2014


### Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

#### RACMO monthly means over the 1985-2014 period

In [3]:
# read RACMO and calculate monthly means over the bias correction period
ds_racmo = xr.open_dataset(racmo_runoff_file,decode_times=False,chunks={"time": 100})

# get indices for bias correction period based on months since 1850-01-15
start_month = (start_year - 1850) * 12
end_month = (end_year - 1850 + 1) * 12 - 1
start_index = np.where(ds_racmo["time"]==start_month)[0][0]
end_index = np.where(ds_racmo["time"]==end_month)[0][0]

# subset racmo to bias correction period
ds_racmo_sel = ds_racmo.isel(time=slice(start_index, end_index + 1))

# add month (1-12) to the dataset
months = ((ds_racmo_sel.time % 12) + 1).astype(np.int32)
ds_racmo_sel = ds_racmo_sel.assign_coords(month=("time", months.data))

# compute mean discharge for each month
racmo_monthly_mean = ds_racmo_sel["Q"].groupby("month").mean(dim="time")

In [4]:
# plot the racmo mean monthly discharge over the bias correction period as a check
""" plotting = 1
if plotting:
    fig, axes = plt.subplots(2, 6, figsize=(12, 6), sharex=True, sharey=True)
    for i in range(12):
        ax = axes[i // 6, i % 6]
        im = ax.imshow(racmo_monthly_mean.isel(month=i).values, cmap='viridis', origin='lower')
        ax.set_title(f'Month: {racmo_monthly_mean.month.values[i]:02d}')
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show() """

" plotting = 1\nif plotting:\n    fig, axes = plt.subplots(2, 6, figsize=(12, 6), sharex=True, sharey=True)\n    for i in range(12):\n        ax = axes[i // 6, i % 6]\n        im = ax.imshow(racmo_monthly_mean.isel(month=i).values, cmap='viridis', origin='lower')\n        ax.set_title(f'Month: {racmo_monthly_mean.month.values[i]:02d}')\n        ax.set_xticks([])\n        ax.set_yticks([])\n        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    plt.tight_layout()\n    plt.show() "

In [5]:
# now do the same for the CMIP model
ds_hist = xr.open_dataset(hist_file,decode_times=False,chunks={"time": 100})
ds_proj = xr.open_dataset(proj_file,decode_times=False,chunks={"time": 100})
ds_ext = xr.open_dataset(ext_file,decode_times=False,chunks={"time": 100})

# concatenate the datasets
ds_cmip = xr.concat([ds_hist, ds_proj, ds_ext], dim="time")

# get indices for bias correction period based on months since 1850-01-15
start_month = (start_year - 1850) * 12
end_month = (end_year - 1850 + 1) * 12 - 1
start_index = np.where(ds_cmip["time"]==start_month)[0][0]
end_index = np.where(ds_cmip["time"]==end_month)[0][0]

# subset cmip to bias correction period
ds_cmip_sel = ds_cmip.isel(time=slice(start_index, end_index + 1))

# add month (1-12) to the dataset
months = ((ds_cmip_sel.time % 12) + 1).astype(np.int32)
ds_cmip_sel = ds_cmip_sel.assign_coords(month=("time", months.data))

# compute mean discharge for each month (takes a few mins)
cmip_monthly_mean = ds_cmip_sel["Q"].groupby("month").mean(dim="time")

In [6]:
# plot the cmip mean monthly discharge over the bias correction period as a check
""" plotting = 1
if plotting:
    fig, axes = plt.subplots(2, 6, figsize=(12, 6), sharex=True, sharey=True)
    for i in range(12):
        ax = axes[i // 6, i % 6]
        im = ax.imshow(cmip_monthly_mean.isel(month=i).values, cmap='viridis', origin='lower')
        ax.set_title(f'Month: {cmip_monthly_mean.month.values[i]:02d}')
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show() """

" plotting = 1\nif plotting:\n    fig, axes = plt.subplots(2, 6, figsize=(12, 6), sharex=True, sharey=True)\n    for i in range(12):\n        ax = axes[i // 6, i % 6]\n        im = ax.imshow(cmip_monthly_mean.isel(month=i).values, cmap='viridis', origin='lower')\n        ax.set_title(f'Month: {cmip_monthly_mean.month.values[i]:02d}')\n        ax.set_xticks([])\n        ax.set_yticks([])\n        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    plt.tight_layout()\n    plt.show() "

In [7]:
# calculate bias as the difference between CMIP and RACMO monthly means
bias = cmip_monthly_mean - racmo_monthly_mean

In [8]:
# plot the bias
""" plotting = 1
if plotting:
    fig, axes = plt.subplots(2, 6, figsize=(12, 6), sharex=True, sharey=True)
    for i in range(12):
        ax = axes[i // 6, i % 6]
        bias_i = bias.isel(month=i).values
        cval = np.nanmax(np.abs(bias_i))
        im = ax.imshow(bias_i, cmap='coolwarm', origin='lower', vmin=-cval, vmax=cval)
        ax.set_title(f'Month: {bias.month.values[i]:02d}')
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show() """

" plotting = 1\nif plotting:\n    fig, axes = plt.subplots(2, 6, figsize=(12, 6), sharex=True, sharey=True)\n    for i in range(12):\n        ax = axes[i // 6, i % 6]\n        bias_i = bias.isel(month=i).values\n        cval = np.nanmax(np.abs(bias_i))\n        im = ax.imshow(bias_i, cmap='coolwarm', origin='lower', vmin=-cval, vmax=cval)\n        ax.set_title(f'Month: {bias.month.values[i]:02d}')\n        ax.set_xticks([])\n        ax.set_yticks([])\n        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    plt.tight_layout()\n    plt.show() "

In [9]:
# subtract the monthly bias from the cmip datasets
# copy of datasets
ds_hist_corrected = ds_hist.copy()
ds_proj_corrected = ds_proj.copy()
ds_ext_corrected = ds_ext.copy()
# add month (1-12) to the datasets
months = ((ds_hist_corrected.time % 12) + 1).astype(np.int32)
ds_hist_corrected = ds_hist_corrected.assign_coords(month=("time", months.data))
months = ((ds_proj_corrected.time % 12) + 1).astype(np.int32)
ds_proj_corrected = ds_proj_corrected.assign_coords(month=("time", months.data))
months = ((ds_ext_corrected.time % 12) + 1).astype(np.int32)
ds_ext_corrected = ds_ext_corrected.assign_coords(month=("time", months.data))
# apply the bias correction to the datasets
ds_hist_corrected["Q"] = ds_hist_corrected["Q"].groupby("month") - bias
ds_proj_corrected["Q"] = ds_proj_corrected["Q"].groupby("month") - bias
ds_ext_corrected["Q"] = ds_ext_corrected["Q"].groupby("month") - bias
# ensure no Q values are negative
ds_hist_corrected["Q"] = ds_hist_corrected["Q"].where(ds_hist_corrected["Q"] > 0, 0)
ds_proj_corrected["Q"] = ds_proj_corrected["Q"].where(ds_proj_corrected["Q"] > 0, 0)
ds_ext_corrected["Q"] = ds_ext_corrected["Q"].where(ds_ext_corrected["Q"] > 0, 0)
# now remove month coordinate from the datasets (in case it gets confused with time)
ds_hist_corrected = ds_hist_corrected.drop_vars("month")
ds_proj_corrected = ds_proj_corrected.drop_vars("month")
ds_ext_corrected = ds_ext_corrected.drop_vars("month")

/Users/dslater2/anaconda3/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 165 times more chunks
  return self.array[key]
/Users/dslater2/anaconda3/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 86 times more chunks
  return self.array[key]
/Users/dslater2/anaconda3/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 199 times more chunks
  return self.array[key]


In [10]:
# define compression settings
compression_settings_Q = dict(zlib=True, complevel=9, shuffle=True, _FillValue=None)
compression_settings_t = dict(_FillValue=None)

# save the corrected historical file (takes around 25 mins)
output_file_hist = hist_file.replace('.nc', '_biascorrected.nc')
ds_hist_corrected.to_netcdf(output_file_hist, mode='w', format='NETCDF4', engine='netcdf4', encoding={"Q": compression_settings_Q, "time": compression_settings_t})

In [12]:
# save the corrected projected file (takes around 10 mins)
output_file_proj = proj_file.replace('.nc', '_biascorrected.nc')
ds_proj_corrected.to_netcdf(output_file_proj, mode='w', format='NETCDF4', engine='netcdf4', encoding={"Q": compression_settings_Q, "time": compression_settings_t})

In [13]:
# save the corrected extended file (takes around 30 mins)
output_file_ext = ext_file.replace('.nc', '_biascorrected.nc')
ds_ext_corrected.to_netcdf(output_file_ext, mode='w', format='NETCDF4', engine='netcdf4', encoding={"Q": compression_settings_Q, "time": compression_settings_t})

In [14]:
# save the bias correction factors (takes a few mins)
bias.to_netcdf(bias_file, mode='w', format='NETCDF4', engine='netcdf4', encoding={"Q": compression_settings_Q})